# KONEPS Procurement: 5-Minute Market Overview

A fast first look at **South Korea's KONEPS public procurement market** using the 2025-09 to 2026-08 release.

This notebook deliberately starts with the compact `01_tenders.parquet` and `03_award_outcomes.parquet` tables, so the first useful result appears quickly. The full dataset also includes 35.9M bidder submissions, contracts, pseudonymized suppliers, agencies, and a tender-contract bridge.

**Privacy:** supplier company names and raw/masked business registration numbers are excluded from the public release. Supplier identities use stable HMAC-SHA256 IDs.  
**Source:** Public Procurement Service via the data.go.kr KONEPS Public Data Open Standard Service.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATASET_SLUG = "koneps-public-procurement-intelligence"
kaggle_input = Path("/kaggle/input")
kaggle_matches = sorted(kaggle_input.rglob("01_tenders.parquet")) if kaggle_input.exists() else []

if kaggle_matches:
    DATA_DIR = kaggle_matches[0].parent.resolve()
else:
    local_candidates = [
        Path("../../data/processed/kaggle_release_202509_202608"),
        Path("data/processed/kaggle_release_202509_202608"),
    ]
    DATA_DIR = next((candidate.resolve() for candidate in local_candidates if candidate.exists()), None)

if DATA_DIR is None:
    available = sorted(str(p) for p in kaggle_input.glob("*") if kaggle_input.exists())
    raise FileNotFoundError(f"KONEPS dataset directory not found; /kaggle/input entries={available}")

print(f"Reading from: {DATA_DIR}")

## Fastest path: one CSV

Dataset V2 adds `00_quickstart_tender_summary.csv` for users who want to start with one flat file. It keeps **one row per tender** (470,937 rows), adds readable English business/contract-method groups, and summarizes linked selected-award and contract activity without exposing supplier identifiers.

```python
quick = pd.read_csv(
    DATA_DIR / "00_quickstart_tender_summary.csv",
    parse_dates=["bid_notice_date", "first_award_date", "last_award_date",
                 "first_contract_date", "last_contract_date"],
)
quick.head()
```

The analysis below still uses projected Parquet columns because that is faster and preserves the canonical relational tables for reproducible joins.

## 1. Load only the columns we need

Column projection keeps this overview light: the two source files are about 65 MB combined in the public release.

In [ ]:
tender_cols = [
    "bid_notice_no", "bid_notice_round", "bid_notice_date",
    "business_div_name_ko", "contract_method_ko",
    "notice_agency_name_ko", "estimated_price_krw",
]
award_cols = [
    "bid_notice_no", "bid_notice_round", "award_rate",
    "award_amount_krw", "opening_date",
]

tenders = pd.read_parquet(DATA_DIR / "01_tenders.parquet", columns=tender_cols)
awards = pd.read_parquet(DATA_DIR / "03_award_outcomes.parquet", columns=award_cols)

print(f"Tenders: {len(tenders):,}")
print(f"Selected award outcomes: {len(awards):,}")
print(f"Tender date range: {tenders['bid_notice_date'].min().date()} to {tenders['bid_notice_date'].max().date()}")

## 2. Market mix: services, construction, goods, foreign supplies

The release preserves KONEPS's Korean business-division labels. For chart readability, common Korean labels are grouped with whitespace-tolerant contains/regex matching; any future unmatched non-empty label is preserved rather than collapsed into a generic bucket.

In [ ]:
def categorize_korean_label(value, rules):
    if pd.isna(value):
        return "Unknown"
    raw = str(value).strip()
    normalized = re.sub(r"\s+", "", raw)
    if not normalized:
        return "Unknown"
    for mode, pattern, category in rules:
        if mode == "prefix" and normalized.startswith(pattern):
            return category
        if mode == "contains" and pattern in normalized:
            return category
        if mode == "regex" and re.search(pattern, normalized):
            return category
    return raw

BUSINESS_DIVISION_RULES = [
    ("contains", "용역", "Services"),
    ("contains", "공사", "Construction"),
    ("contains", "물품", "Goods"),
    ("contains", "외자", "Foreign supplies"),
]

business_division = tenders["business_div_name_ko"].map(
    lambda value: categorize_korean_label(value, BUSINESS_DIVISION_RULES)
)

mix = (
    business_division
    .value_counts()
    .rename_axis("business_division")
    .to_frame("tenders")
)
mix["share_pct"] = 100 * mix["tenders"] / mix["tenders"].sum()
display(mix)

ax = mix["tenders"].sort_values().plot(kind="barh", figsize=(8, 4), legend=False)
ax.set_title("KONEPS tender notices by business division")
ax.set_xlabel("Tender notices")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 3. Monthly tender volume

This is a publication-volume view, not seasonally adjusted demand. It is useful for spotting calendar effects before building predictive models.

In [ ]:
monthly = (
    tenders.assign(month=tenders["bid_notice_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)
    .size()
    .rename(columns={"size": "tenders"})
)

display(monthly)
ax = monthly.plot(x="month", y="tenders", marker="o", figsize=(9, 4), legend=False)
ax.set_title("Monthly KONEPS tender notices")
ax.set_xlabel("")
ax.set_ylabel("Tender notices")
plt.tight_layout()
plt.show()

## 4. Award-rate comparison by business division

`award_rate` is the value reported by KONEPS. We keep only finite-looking values from 0 to 100 for this descriptive view and join selected awards back to tender attributes on `(bid_notice_no, bid_notice_round)`. Source anomalies are preserved in the dataset rather than silently rewritten.

In [ ]:
valid_awards = awards.loc[awards["award_rate"].between(0, 100, inclusive="both")].copy()
joined = valid_awards.merge(
    tenders[["bid_notice_no", "bid_notice_round", "business_div_name_ko", "contract_method_ko"]],
    on=["bid_notice_no", "bid_notice_round"],
    how="inner",
    validate="many_to_one",
)
joined["business_division"] = joined["business_div_name_ko"].map(
    lambda value: categorize_korean_label(value, BUSINESS_DIVISION_RULES)
)

overall_median = joined["award_rate"].median()
summary = (
    joined.groupby("business_division")["award_rate"]
    .agg(observations="size", median="median", mean="mean")
    .sort_values("observations", ascending=False)
)
print(f"In-scope valid selected awards: {len(joined):,}")
print(f"Overall median reported award rate: {overall_median:.3f}%")
display(summary)

ax = summary.sort_values("median")["median"].plot(kind="barh", figsize=(8, 4), legend=False)
ax.set_title("Median reported award rate by business division")
ax.set_xlabel("Award rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 5. Contracting method matters

A simple grouped view shows why procurement method should be controlled for in any price or award-rate model. These are descriptive associations, not causal effects.

In [ ]:
CONTRACT_METHOD_RULES = [
    ("contains", "수의", "Direct / negotiated"),
    ("regex", "제한.*경쟁|경쟁.*제한", "Restricted competition"),
    ("regex", "일반.*경쟁|경쟁.*일반", "Open competition"),
    ("regex", "지명.*경쟁|경쟁.*지명", "Selective competition"),
]
joined["contract_method"] = joined["contract_method_ko"].map(
    lambda value: categorize_korean_label(value, CONTRACT_METHOD_RULES)
)

method_summary = (
    joined.groupby("contract_method")["award_rate"]
    .agg(observations="size", median="median")
    .sort_values("observations", ascending=False)
    .head(10)
)
display(method_summary)

ax = method_summary.sort_values("median")["median"].plot(kind="barh", figsize=(8, 4), legend=False)
ax.set_title("Median reported award rate by contract method")
ax.set_xlabel("Award rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 6. Which agencies publish the most tenders?

Agency names are public institutional identifiers and are intentionally retained in the release.

In [ ]:
top_agencies = (
    tenders["notice_agency_name_ko"]
    .dropna()
    .value_counts()
    .head(15)
    .rename_axis("notice_agency_name_ko")
    .to_frame("tenders")
)
display(top_agencies)

## Where to go next

The relational release is designed for deeper work without changing grain:

- `02_bidder_submissions.parquet` ? **35.9M** bidder-level submissions for competition intensity, repeated participation, and bid clustering.
- `04_contracts.parquet` ? executed contracts and amounts.
- `05_suppliers.parquet` ? privacy-minimized supplier roles and full-window activity snapshots. Treat `snapshot_total_*` as descriptive full-window aggregates, **not leak-free historical features**.
- `06_agencies.parquet` ? agency dimension.
- `07_tender_contract_bridge.parquet` ? explicit tender-to-contract relationship table.

For prediction, construct all historical supplier/agency features using only information available before the target observation date. The next useful notebook is a time-safe competition baseline built from bidder submissions.